## 1. Data Loading and Initial Inspection

In [ ]:
import pandas as pd

df = pd.read_csv("/content/TASK 1.csv")

# Numeric columns worth analyzing
print(df[['Quantity', 'UnitPrice', 'TotalPrice', 'ItemsInCart']].describe())

          Quantity    UnitPrice   TotalPrice  ItemsInCart
count  1200.000000  1200.000000  1200.000000  1200.000000
mean      2.945833   356.412750  1053.968300     5.485000
std       1.407557   197.177146   819.856558     2.281983
min       1.000000    11.390000    11.390000     1.000000
25%       2.000000   186.062500   410.520000     4.000000
50%       3.000000   364.210000   823.615000     5.000000
75%       4.000000   521.570000  1578.475000     7.000000
max       5.000000   699.930000  3456.400000    10.000000


## 2. Outlier Detection using IQR

In [ ]:
for col in ['TotalPrice', 'Quantity', 'UnitPrice']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"\n{col}: Q1={Q1}, Q3={Q3}, IQR={IQR}")
    print(f"  Bounds: [{lower:.2f}, {upper:.2f}]")
    print(f"  Outliers found: {len(outliers)}")
    print(outliers[['OrderID', col]].head(5))


TotalPrice: Q1=410.52, Q3=1578.475, IQR=1167.955
  Bounds: [-1341.41, 3330.41]
  Outliers found: 8
       OrderID  TotalPrice
107  ORD200107     3353.75
326  ORD200326     3352.40
328  ORD200328     3370.20
469  ORD200469     3384.90
632  ORD200632     3390.80

Quantity: Q1=2.0, Q3=4.0, IQR=2.0
  Bounds: [-1.00, 7.00]
  Outliers found: 0
Empty DataFrame
Columns: [OrderID, Quantity]
Index: []

UnitPrice: Q1=186.0625, Q3=521.5699999999999, IQR=335.50749999999994
  Bounds: [-317.20, 1024.83]
  Outliers found: 0
Empty DataFrame
Columns: [OrderID, UnitPrice]
Index: []


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Best selling products
print(df['Product'].value_counts())

# Revenue by product
print(df.groupby('Product')['TotalPrice'].sum().sort_values(ascending=False))

# Orders by payment method
print(df['PaymentMethod'].value_counts())

# Orders by status
print(df['OrderStatus'].value_counts())

# Monthly trend
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.to_period('M')
print(df.groupby('Month')['TotalPrice'].sum())

Product
Printer    181
Tablet     179
Chair      178
Laptop     173
Desk       170
Monitor    163
Phone      156
Name: count, dtype: int64
Product
Chair      195620.11
Printer    195612.61
Laptop     192126.56
Tablet     186568.95
Monitor    175651.41
Desk       167459.93
Phone      151722.39
Name: TotalPrice, dtype: float64
PaymentMethod
Online         258
Cash           246
Credit Card    234
Debit Card     232
Gift Card      230
Name: count, dtype: int64
OrderStatus
Cancelled    250
Returned     247
Pending      237
Shipped      235
Delivered    231
Name: count, dtype: int64
Month
2023-01    56685.75
2023-02    40117.66
2023-03    48609.37
2023-04    27751.71
2023-05    63836.84
2023-06    49500.19
2023-07    42820.66
2023-08    54352.14
2023-09    29526.67
2023-10    52607.85
2023-11    43079.67
2023-12    43754.73
2024-01    38528.08
2024-02    36909.57
2024-03    36030.90
2024-04    49613.14
2024-05    27909.11
2024-06    68068.54
2024-07    42963.98
2024-08    31991.07
2024-09  

## 4. Correlation Analysis

In [ ]:
print(df[['Quantity', 'UnitPrice', 'TotalPrice', 'ItemsInCart']].corr())

             Quantity  UnitPrice  TotalPrice  ItemsInCart
Quantity     1.000000   0.014553    0.615251     0.650061
UnitPrice    0.014553   1.000000    0.717081     0.000602
TotalPrice   0.615251   0.717081    1.000000     0.392540
ItemsInCart  0.650061   0.000602    0.392540     1.000000


## 5. EDA Summary and Recommendations

In [ ]:
print("""
=== EDA SUMMARY: KEY FINDINGS ===

1. PROBLEM: Understand sales patterns, revenue drivers, and risk areas.

2. METHODOLOGY: Descriptive stats, IQR outlier detection, trend analysis, correlation.

3. KEY FINDINGS:
   - 8 high-value outlier orders (>$3,330) identified — likely bulk buyers, not errors.
   - Chair generates highest revenue ($195,620) despite not being top ordered product.
   - Phone is weakest product: lowest orders (156) and lowest revenue ($151,722).
   - CRITICAL: 41% of orders are Cancelled or Returned — major revenue leakage.
   - Peak month: June 2024 ($68,068). Lowest: April 2024 ($27,909).
   - UnitPrice is the strongest driver of TotalPrice (r=0.72).

4. RECOMMENDATIONS:
   - Investigate cancellation/return reasons urgently — 41% is abnormally high.
   - Consider promotions on Phone to boost its performance.
   - Retain and reward the 8 bulk-buying customers — they are VIP accounts.
   - Investigate what drove the June 2024 spike to replicate it.
""")


=== EDA SUMMARY: KEY FINDINGS ===

1. PROBLEM: Understand sales patterns, revenue drivers, and risk areas.

2. METHODOLOGY: Descriptive stats, IQR outlier detection, trend analysis, correlation.

3. KEY FINDINGS:
   - 8 high-value outlier orders (>$3,330) identified — likely bulk buyers, not errors.
   - Chair generates highest revenue ($195,620) despite not being top ordered product.
   - Phone is weakest product: lowest orders (156) and lowest revenue ($151,722).
   - CRITICAL: 41% of orders are Cancelled or Returned — major revenue leakage.
   - Peak month: June 2024 ($68,068). Lowest: April 2024 ($27,909).
   - UnitPrice is the strongest driver of TotalPrice (r=0.72).

4. RECOMMENDATIONS:
   - Investigate cancellation/return reasons urgently — 41% is abnormally high.
   - Consider promotions on Phone to boost its performance.
   - Retain and reward the 8 bulk-buying customers — they are VIP accounts.
   - Investigate what drove the June 2024 spike to replicate it.

